# 🚀 Notebook 07 – Final Pipeline (Demo Presentasi)
## Lettuce Fresh Weight Prediction | ANN + PSO Improved

> **Notebook ini adalah demo end-to-end untuk presentasi.**  
> Menggunakan model terbaik hasil `07_Improved_ANN_PSO.ipynb`.
> Jika model belum ada, jalankan NB08 terlebih dahulu.

### Alur:
```
Gambar → Segmentasi HSV → Ekstrak 17 Fitur → Seleksi 10 Fitur
       → Log Transform → Scaler → ANN+PSO Improved → Prediksi Berat
```

### Hasil Model Terbaik (5-Fold CV):
| Metrik | ANN Base | ANN+PSO | **ANN Improved** |
|--------|----------|---------|------------------|
| MAE    | 4.62 g   | 4.03 g  | **2.97 g**       |
| RMSE   | 5.72 g   | 5.09 g  | **3.80 g**       |
| R²     | -0.028   | 0.100   | **0.556**        |
| MAPE   | 15.51%   | 13.24%  | **9.66%**        |

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os, joblib, warnings
warnings.filterwarnings('ignore')

from scipy.stats import skew
from skimage.feature import graycomatrix, graycoprops
from scipy.stats import entropy as scipy_entropy
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams.update({
    'figure.facecolor': '#1a1a2e', 'axes.facecolor': '#16213e',
    'axes.labelcolor': 'white', 'xtick.color': 'white',
    'ytick.color': 'white', 'text.color': 'white',
    'axes.edgecolor': '#0f3460', 'grid.color': '#0f3460', 'axes.grid': True,
})
print('✅ Libraries loaded!')

---
## ⚙️ Konfigurasi

In [ ]:
BASE_DIR  = r'D:/laragon/www/lettuce-weight-prediction'
IMAGE_DIR = f'{BASE_DIR}/data/raw/images'
CSV_PATH  = f'{BASE_DIR}/data/raw/measurements_raw.csv'

LOWER_GREEN = np.array([20, 30, 30])
UPPER_GREEN = np.array([95, 255, 255])

ALL_FEATURES = [
    'leaf_area', 'perimeter', 'aspect_ratio', 'extent', 'equivalent_diameter',
    'mean_R', 'mean_G', 'mean_B', 'mean_H', 'mean_S', 'mean_V', 'std_G',
    'contrast', 'energy', 'homogeneity', 'correlation', 'entropy'
]

os.makedirs(f'{BASE_DIR}/output/plots', exist_ok=True)
os.makedirs(f'{BASE_DIR}/output/models', exist_ok=True)

# ── Load model & metadata ────────────────────────────────────
model_path = f'{BASE_DIR}/output/models/ann_model_improved.pkl'
meta_path  = f'{BASE_DIR}/output/models/improved_metadata.pkl'
scaler_path = f'{BASE_DIR}/output/models/scaler_improved.pkl'

assert os.path.exists(model_path),  '❌ Model belum ada! Jalankan 07_Improved_ANN_PSO.ipynb dulu.'
assert os.path.exists(meta_path),   '❌ Metadata belum ada! Jalankan 07_Improved_ANN_PSO.ipynb dulu.'
assert os.path.exists(scaler_path), '❌ Scaler belum ada! Jalankan 07_Improved_ANN_PSO.ipynb dulu.'

model  = joblib.load(model_path)
meta   = joblib.load(meta_path)
scaler = joblib.load(scaler_path)

SELECTED_FEATURES = meta['selected_features']
USE_LOG           = meta['use_log']
BEST_LAYERS       = tuple(meta['layers'])

print('✅ Model & konfigurasi berhasil dimuat!')
print(f'   Arsitektur    : {BEST_LAYERS}')
print(f'   Fitur dipakai : {len(SELECTED_FEATURES)} dari {len(ALL_FEATURES)}')
print(f'   Log Transform : {USE_LOG}')
print(f'   Scaler        : {meta["scaler"]}')

---
## 🔬 Helper: Segmentasi & Ekstraksi Fitur

In [ ]:
def segment_lettuce(img_bgr):
    """Segmentasi area hijau selada menggunakan threshold HSV."""
    # Mask daun hijau + akar gelap (V<65, S<80 = tanah)
    hsv  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    mask_g = cv2.inRange(hsv, LOWER_GREEN, UPPER_GREEN)
    mask_r = cv2.inRange(hsv, np.array([0, 0, 5]), np.array([180, 80, 75]))
    mask = cv2.bitwise_or(mask_g, mask_r)
    k_o  = np.ones((3,3), np.uint8)
    k_c  = np.ones((7,7), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k_o, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k_c, iterations=3)
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if cnts:
        clean = np.zeros_like(mask)
        cv2.drawContours(clean, [max(cnts, key=cv2.contourArea)], -1, 255, cv2.FILLED)
        mask = cv2.morphologyEx(clean, cv2.MORPH_CLOSE, k_c, iterations=5)
    return mask, cv2.bitwise_and(img_bgr, img_bgr, mask=mask)


def extract_features(img_bgr, mask):
    """Ekstrak semua 17 fitur dari gambar + mask."""
    feat = {}
    # Morfologi
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if cnts:
        c = max(cnts, key=cv2.contourArea)
        area = cv2.contourArea(c)
        x, y, w, h = cv2.boundingRect(c)
        feat['leaf_area']           = float(area)
        feat['perimeter']           = float(cv2.arcLength(c, True))
        feat['aspect_ratio']        = float(w/h) if h > 0 else 0.0
        feat['extent']              = float(area/(w*h)) if w*h > 0 else 0.0
        feat['equivalent_diameter'] = float(np.sqrt(4*area/np.pi)) if area > 0 else 0.0
    else:
        for k in ['leaf_area','perimeter','aspect_ratio','extent','equivalent_diameter']:
            feat[k] = 0.0
    # Warna
    mb = mask > 0
    if mb.any():
        rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
        feat['mean_R'] = float(rgb[:,:,0][mb].mean())
        feat['mean_G'] = float(rgb[:,:,1][mb].mean())
        feat['mean_B'] = float(rgb[:,:,2][mb].mean())
        feat['std_G']  = float(rgb[:,:,1][mb].std())
        feat['mean_H'] = float(hsv[:,:,0][mb].mean())
        feat['mean_S'] = float(hsv[:,:,1][mb].mean())
        feat['mean_V'] = float(hsv[:,:,2][mb].mean())
    else:
        for k in ['mean_R','mean_G','mean_B','std_G','mean_H','mean_S','mean_V']:
            feat[k] = 0.0
    # Tekstur GLCM
    try:
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        gm   = cv2.bitwise_and(gray, gray, mask=mask)
        cnts2, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        x, y, w, h = cv2.boundingRect(max(cnts2, key=cv2.contourArea))
        roi  = gm[y:y+h, x:x+w]
        q    = (roi // 32).astype(np.uint8)
        glcm = graycomatrix(q, [1], [0, np.pi/4, np.pi/2, 3*np.pi/4],
                             levels=8, symmetric=True, normed=True)
        feat['contrast']    = float(np.mean(graycoprops(glcm, 'contrast')))
        feat['energy']      = float(np.mean(graycoprops(glcm, 'energy')))
        feat['homogeneity'] = float(np.mean(graycoprops(glcm, 'homogeneity')))
        feat['correlation'] = float(np.mean(graycoprops(glcm, 'correlation')))
        h_arr = np.histogram(roi[roi>0], bins=256, range=(0,256))[0]
        feat['entropy'] = float(scipy_entropy(h_arr/(h_arr.sum()+1e-10) + 1e-10))
    except:
        for k in ['contrast','energy','homogeneity','correlation','entropy']:
            feat[k] = 0.0
    return feat


def predict_weight(image_path):
    """
    Prediksi berat selada dari path gambar menggunakan model terbaik.
    Returns: (predicted_gram, mask, segmented_img, feature_dict)
    """
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f'Gambar tidak dapat dibaca: {image_path}')
    img  = cv2.resize(img, (640, 480))
    mask, seg = segment_lettuce(img)
    feat = extract_features(img, mask)

    # Pilih hanya fitur yang digunakan model
    X = np.array([feat[f] for f in SELECTED_FEATURES]).reshape(1, -1)
    X_s = scaler.transform(X)

    # Prediksi & inverse transform
    y_pred = model.predict(X_s)[0]
    if USE_LOG:
        y_pred = np.expm1(y_pred)

    return max(0.0, y_pred), mask, seg, feat


print('✅ Helper functions siap!')

---
## 📦 Load Dataset & Tampilkan Sampel

In [ ]:
df_csv = pd.read_csv(CSV_PATH, sep=';')
df_csv.columns = df_csv.columns.str.strip()

valid = []
for _, row in df_csv.iterrows():
    p = f"{IMAGE_DIR}/{row['foto']}"
    if os.path.exists(p):
        valid.append({
            'id': row['id_tanaman'],
            'foto': row['foto'],
            'image_path': p,
            'weight_gram': float(row['bobot_segar_gram'])
        })
df_valid = pd.DataFrame(valid)

print(f'✅ Dataset: {len(df_valid)} gambar')
print(f'   Berat: {df_valid["weight_gram"].min():.0f}g – {df_valid["weight_gram"].max():.0f}g')
print(f'   Rata-rata: {df_valid["weight_gram"].mean():.1f}g')

# Tampilkan 10 sampel (sorted by weight)
sorted_df = df_valid.sort_values('weight_gram').reset_index(drop=True)
idx_show  = np.linspace(0, len(sorted_df)-1, 10, dtype=int)

fig, axes = plt.subplots(2, 5, figsize=(25, 10))
fig.suptitle('🥬 Dataset Selada – Sampel Gambar (Terurut Berat)',
             fontsize=16, fontweight='bold', color='white')

for ax, i in zip(axes.flatten(), idx_show):
    row = sorted_df.iloc[i]
    img = cv2.cvtColor(cv2.imread(row['image_path']), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f'{row["id"]}\n{row["weight_gram"]:.0f}g',
                  fontsize=10, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig(f'{BASE_DIR}/output/plots/dataset_samples.png', dpi=150,
             bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('✅ Simpan → output/plots/dataset_samples.png')

---
## 🔬 Visualisasi Segmentasi

In [ ]:
samples = df_valid.sample(6, random_state=7).reset_index(drop=True)

fig, axes = plt.subplots(2, 6, figsize=(28, 10))
fig.suptitle('🔬 Proses Segmentasi HSV – Contoh Gambar',
             fontsize=14, fontweight='bold', color='white')

for col, (_, row) in enumerate(samples.iterrows()):
    img = cv2.resize(cv2.imread(row['image_path']), (640, 480))
    mask, seg = segment_lettuce(img)

    axes[0, col].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0, col].set_title(f'{row["id"]} – {row["weight_gram"]:.0f}g', fontsize=9)
    axes[0, col].axis('off')

    axes[1, col].imshow(cv2.cvtColor(seg, cv2.COLOR_BGR2RGB))
    cover = np.count_nonzero(mask) / mask.size * 100
    axes[1, col].set_title(f'Coverage: {cover:.1f}%', fontsize=9)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=11, color='white')
axes[1, 0].set_ylabel('Segmented', fontsize=11, color='white')

plt.tight_layout()
plt.savefig(f'{BASE_DIR}/output/plots/segmentation_results.png', dpi=150,
             bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('✅ Simpan → output/plots/segmentation_results.png')

---
## 🔮 Demo Prediksi – Model Terbaik

In [ ]:
demo_samples = df_valid.sample(8, random_state=42).reset_index(drop=True)

results = []
for _, row in demo_samples.iterrows():
    try:
        pred, mask, seg, feat = predict_weight(row['image_path'])
        results.append({
            'id':      row['id'],
            'actual':  row['weight_gram'],
            'pred':    round(pred, 2),
            'error':   round(abs(row['weight_gram'] - pred), 2),
            'mape':    round(abs(row['weight_gram'] - pred) / row['weight_gram'] * 100, 1),
            'img_path':row['image_path'],
            'seg':     seg,
        })
    except Exception as e:
        print(f'  ⚠️ Skip {row["id"]}: {e}')

# Visualisasi grid prediksi
fig, axes = plt.subplots(2, len(results), figsize=(4*len(results), 10))
if len(results) == 1:
    axes = axes.reshape(2, 1)
fig.suptitle('🔮 Demo Prediksi Berat Selada – ANN Improved',
             fontsize=15, fontweight='bold', color='white')

for col, r in enumerate(results):
    color = '#2ECC71' if r['mape'] < 10 else ('#F39C12' if r['mape'] < 20 else '#E74C3C')

    orig = cv2.cvtColor(cv2.imread(r['img_path']), cv2.COLOR_BGR2RGB)
    axes[0, col].imshow(orig)
    axes[0, col].set_title(f"{r['id']}\nAktual: {r['actual']:.0f}g", fontsize=9)
    axes[0, col].axis('off')

    seg_rgb = cv2.cvtColor(r['seg'], cv2.COLOR_BGR2RGB)
    axes[1, col].imshow(seg_rgb)
    axes[1, col].set_title(
        f"Pred: {r['pred']:.1f}g\nError: {r['error']:.1f}g ({r['mape']:.1f}%)",
        fontsize=9, color=color
    )
    axes[1, col].axis('off')

plt.tight_layout()
plt.savefig(f'{BASE_DIR}/output/plots/demo_predictions.png', dpi=150,
             bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

# Tabel ringkasan
df_res = pd.DataFrame([{k: r[k] for k in ['id','actual','pred','error','mape']} for r in results])
print('\n📋 Tabel Prediksi:')
print(df_res.to_string(index=False))
print(f'\n   Rata-rata |Error| : {df_res["error"].mean():.2f}g')
print(f'   Rata-rata MAPE    : {df_res["mape"].mean():.1f}%')

---
## 📊 Dashboard Evaluasi Model

In [ ]:
# Prediksi seluruh dataset
all_actual, all_pred = [], []
for _, row in df_valid.iterrows():
    try:
        pred, _, _, _ = predict_weight(row['image_path'])
        all_actual.append(row['weight_gram'])
        all_pred.append(pred)
    except:
        pass

y_true = np.array(all_actual)
y_pred = np.array(all_pred)

mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2   = r2_score(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100
resid = y_true - y_pred

# Dashboard 2x2
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle('📊 Dashboard Evaluasi – ANN Improved (Seluruh Dataset)',
             fontsize=15, fontweight='bold', color='white')

# 1. Actual vs Predicted
err_c = np.abs(y_true - y_pred)
sc = axes[0,0].scatter(y_true, y_pred, c=err_c, cmap='RdYlGn_r',
                        s=70, alpha=0.8, edgecolors='white', lw=0.4)
plt.colorbar(sc, ax=axes[0,0], label='|Error| gram')
lim = [min(y_true.min(), y_pred.min())*0.92, max(y_true.max(), y_pred.max())*1.08]
axes[0,0].plot(lim, lim, 'w--', lw=2, label='Perfect Prediction')
axes[0,0].set_title(f'Actual vs Predicted  |  R²={r2:.4f}', fontsize=11, fontweight='bold')
axes[0,0].set_xlabel('Berat Aktual (gram)'); axes[0,0].set_ylabel('Berat Prediksi (gram)')
axes[0,0].legend(fontsize=9)

# 2. Residual Plot
axes[0,1].scatter(y_pred, resid, c=resid, cmap='coolwarm',
                   s=60, alpha=0.8, edgecolors='white', lw=0.4)
axes[0,1].axhline(0, color='white', ls='--', lw=2, label='Zero')
axes[0,1].axhline(resid.mean(), color='#F39C12', lw=1.5,
                   label=f'Mean: {resid.mean():.3f}')
axes[0,1].set_title('Residual Plot', fontsize=11, fontweight='bold')
axes[0,1].set_xlabel('Prediksi (gram)'); axes[0,1].set_ylabel('Residual (gram)')
axes[0,1].legend(fontsize=9)

# 3. Residual Distribution
axes[1,0].hist(resid, bins=25, color='#9B59B6', edgecolor='white', alpha=0.85)
axes[1,0].axvline(0, color='white', ls='--', lw=2)
axes[1,0].axvline(resid.mean(), color='#F39C12', lw=1.5,
                   label=f'Mean: {resid.mean():.2f}')
axes[1,0].set_title('Distribusi Residual', fontsize=11, fontweight='bold')
axes[1,0].set_xlabel('Residual (gram)'); axes[1,0].set_ylabel('Frekuensi')
axes[1,0].legend(fontsize=9)

# 4. Metrik ringkasan
axes[1,1].axis('off')
metrics_txt = [
    ('Model', 'ANN + PSO Improved'),
    ('Fitur', f'{len(SELECTED_FEATURES)} dari {len(ALL_FEATURES)}'),
    ('Arsitektur', str(BEST_LAYERS)),
    ('Dataset', f'{len(y_true)} sampel'),
    ('', ''),
    ('MAE',  f'{mae:.4f} gram'),
    ('RMSE', f'{rmse:.4f} gram'),
    ('R²',   f'{r2:.4f}'),
    ('MAPE', f'{mape:.2f}%'),
]
y_pos = 0.95
for label, val in metrics_txt:
    if label:
        axes[1,1].text(0.05, y_pos, f'{label}:', fontsize=12, color='#95A5A6',
                        transform=axes[1,1].transAxes, va='top')
        axes[1,1].text(0.45, y_pos, val, fontsize=12, color='white', fontweight='bold',
                        transform=axes[1,1].transAxes, va='top')
    y_pos -= 0.09
axes[1,1].set_title('Ringkasan Model', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{BASE_DIR}/output/plots/final_dashboard.png', dpi=150,
             bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('✅ Simpan → output/plots/final_dashboard.png')

---
## ✅ Ringkasan Akhir

In [ ]:
print('=' * 60)
print('🏆 LETTUCE FRESH WEIGHT PREDICTION – HASIL AKHIR')
print('=' * 60)
print()
print('  Pipeline:')
print('  Gambar → Segmentasi HSV → Ekstraksi 17 Fitur')
print('  → Seleksi Fitur → Log Transform → Scaler → ANN+PSO')
print()
print('  Konfigurasi Model Terbaik:')
print(f'    Arsitektur  : {BEST_LAYERS}')
print(f'    LR          : {meta["lr"]:.6f}')
print(f'    Alpha (L2)  : {meta["alpha"]:.6f}')
print(f'    Scaler      : {meta["scaler"]}')
print(f'    Log Target  : {USE_LOG}')
print(f'    Fitur pakai : {len(SELECTED_FEATURES)}')
print()
print('  Performa (Full Dataset):')
print(f'    MAE  : {mae:.4f} gram')
print(f'    RMSE : {rmse:.4f} gram')
print(f'    R²   : {r2:.4f}')
print(f'    MAPE : {mape:.2f}%')
print()
print('  File Output:')
output_files = [
    'output/models/ann_model_improved.pkl',
    'output/models/scaler_improved.pkl',
    'output/models/improved_metadata.pkl',
    'output/plots/final_dashboard.png',
    'output/plots/demo_predictions.png',
    'output/plots/segmentation_results.png',
    'output/plots/dataset_samples.png',
]
for f in output_files:
    exists = '✅' if os.path.exists(f'{BASE_DIR}/{f}') else '❌'
    print(f'    {exists} {f}')
print()
print('🎉 Demo selesai!')